# Long weak-grid inverter run at 50 µs

This notebook plots the 0.2 s run from `long_weak_grid_50us.json`; the shaded region is the unbalanced sag from 80 to 120 ms.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path("/Volumes/SSDMatthias/development/dpsim")
CSV = REPO / "build-macos-check/fmi-validation/long_weak_grid_50us/logs/fmi_mixed.csv"
data = pd.read_csv(CSV, skipinitialspace=True)
data.columns = data.columns.str.strip()
data["time_ms"] = 1000 * data["time"]
data.head()

## PCC and grid voltages

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for p, phase in enumerate("abc"):
    axes[p].plot(data["time_ms"], data["grid_voltage_" + str(p)] / 1000, label="grid")
    axes[p].plot(data["time_ms"], data["pcc_voltage_" + str(p)] / 1000, label="PCC")
    axes[p].axvspan(80, 120, color="tab:red", alpha=0.12)
    axes[p].set_ylabel(phase.upper() + " [kV]")
    axes[p].grid(True, alpha=0.25)
    axes[p].legend()
axes[-1].set_xlabel("time [ms]")
fig.suptitle("Three-phase voltage during weak-grid sag")
fig.tight_layout()

## Converter, line, and load currents

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for p, phase in enumerate("abc"):
    axes[p].plot(
        data["time_ms"], data["converter_current_" + str(p)], label="converter"
    )
    axes[p].plot(data["time_ms"], data["line_current_" + str(p)], label="line")
    axes[p].plot(data["time_ms"], data["load_current_" + str(p)], label="native load")
    axes[p].axvspan(80, 120, color="tab:red", alpha=0.12)
    axes[p].set_ylabel(phase.upper() + " [A]")
    axes[p].grid(True, alpha=0.25)
    axes[p].legend()
axes[-1].set_xlabel("time [ms]")
fig.suptitle("Current sharing and recovery")
fig.tight_layout()

## Multirate and controller diagnostics

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(data["time_ms"], data["fmi_coupling_error_A"])
axes[0].set_ylabel("coupling error [A]")
axes[1].step(data["time_ms"], data["fmi_internal_steps"], where="post")
axes[1].set_ylabel("FMI steps")
axes[2].plot(data["time_ms"], data["pi_d"])
axes[2].plot(data["time_ms"], data["pi_q"])
axes[2].set_ylabel("PI control")
axes[3].plot(data["time_ms"], data["pll_frequency"])
axes[3].set_ylabel("PLL [Hz]")
for ax in axes:
    ax.axvspan(80, 120, color="tab:red", alpha=0.12)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel("time [ms]")
fig.suptitle("Scheduler diagnostics and controller recovery")
fig.tight_layout()

## Summary

In [ ]:
fault = data[data["time"].between(0.08, 0.12, inclusive="left")]
recovery = data[data["time"] >= 0.12]
pd.DataFrame(
    {
        "samples": [len(data), len(fault), len(recovery)],
        "max coupling error [A]": [
            data["fmi_coupling_error_A"].max(),
            fault["fmi_coupling_error_A"].max(),
            recovery["fmi_coupling_error_A"].max(),
        ],
        "internal steps": [
            sorted(data["fmi_internal_steps"].unique()),
            sorted(fault["fmi_internal_steps"].unique()),
            sorted(recovery["fmi_internal_steps"].unique()),
        ],
    },
    index=["full run", "sag", "recovery"],
)